In [1]:
import ee
import pandas as pd

ee.Initialize()

aoi = ee.Geometry.Polygon([
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
])

# --- 1. Global Flood Database (GFD) - Eventos históricos de inundación ---
print("=== Global Flood Database ===")
gfd = ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')
gfd_cgsm = gfd.filterBounds(aoi)
print(f"Eventos de inundación en la CGSM: {gfd_cgsm.size().getInfo()}")

# Listar eventos
events = gfd_cgsm.aggregate_array('system:index').getInfo()
for evt in events:
    img = gfd.filter(ee.Filter.eq('system:index', evt)).first()
    info = img.getInfo()['properties']
    print(f"  Evento: {evt}")
    print(f"    País: {info.get('country', 'N/A')}")
    print(f"    Duración: {info.get('duration_days', 'N/A')} días")
    print(f"    Muertos: {info.get('dead', 'N/A')}")
    print(f"    Desplazados: {info.get('displaced', 'N/A')}")
    print()

# --- 2. JRC Global Surface Water - Cambios en agua superficial ---
print("=== JRC Global Surface Water ===")
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
occurrence = jrc.select('occurrence').clip(aoi)
seasonality = jrc.select('seasonality').clip(aoi)

# Área con agua > 50% del tiempo
water_area = occurrence.gt(50).multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=30,
    maxPixels=1e13
).get('occurrence').getInfo()
print(f"Área con agua > 50% del tiempo: {water_area/1e6:.1f} km²")

# --- 3. Sentinel-1 SAR para detección de inundación en sept 2020 ---
print("\n=== Detección de inundación con Sentinel-1 SAR ===")

# Período seco de referencia (enero-marzo 2020)
s1_dry = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate('2020-01-01', '2020-03-31')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select('VH')
    .median()
    .clip(aoi))

# Período de inundación (septiembre-octubre 2020 - La Niña)
s1_flood = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate('2020-09-01', '2020-10-31')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select('VH')
    .median()
    .clip(aoi))

# Diferencia SAR (inundación = caída en backscatter VH)
sar_diff = s1_dry.subtract(s1_flood).rename('SAR_diff')

# Umbral: diferencia > 3 dB indica inundación
flood_mask = sar_diff.gt(3).selfMask().rename('flood')

# Calcular área inundada
flood_area = flood_mask.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=10,
    maxPixels=1e13
).get('flood').getInfo()

print(f"Imágenes SAR período seco: {ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi).filterDate('2020-01-01', '2020-03-31').filter(ee.Filter.eq('instrumentMode', 'IW')).size().getInfo()}")
print(f"Imágenes SAR período inundación: {ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi).filterDate('2020-09-01', '2020-10-31').filter(ee.Filter.eq('instrumentMode', 'IW')).size().getInfo()}")
print(f"Área inundada (sept-oct 2020): {flood_area/1e6:.1f} km²")

# --- 4. Cruce con anomalías NDVI ---
print("\n=== CRUCE: Inundación SAR vs Anomalías NDVI ===")

stations = {
    'Isla Boquerón': [-74.298, 10.962],
    'Punta Cerro': [-74.283, 10.973],
    'Punta Chino': [-74.305, 10.912],
    'Río Sevilla': [-74.325, 10.880],
    'Caño Palos': [-74.471, 10.758],
    'CP Pajarales': [-74.75, 10.85],
    'Caño Clarín': [-74.55, 10.55],
    'VIPIS': [-74.65, 11.02],
}

print(f"{'Estación':<20} {'SAR diff (dB)':>15} {'Inundada':>10}")
print("-" * 47)
for name, coords in stations.items():
    pt = ee.Geometry.Point(coords).buffer(500)
    diff_val = sar_diff.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pt,
        scale=10,
        maxPixels=1e8
    ).get('SAR_diff').getInfo()
    
    inundada = "SÍ" if diff_val and diff_val > 3 else "NO"
    diff_str = f"{diff_val:.2f}" if diff_val else "N/A"
    print(f"{name:<20} {diff_str:>15} {inundada:>10}")

print("\nNota: SAR diff > 3 dB indica probable inundación")
print("Fuente: Sentinel-1 GRD VH, comparación enero-marzo vs septiembre-octubre 2020")

=== Global Flood Database ===
Eventos de inundación en la CGSM: 16
  Evento: DFO_1818_From_20011026_to_20011115
    País: N/A
    Duración: N/A días
    Muertos: N/A
    Desplazados: N/A

  Evento: DFO_1996_From_20020720_to_20020731
    País: N/A
    Duración: N/A días
    Muertos: N/A
    Desplazados: N/A

  Evento: DFO_2588_From_20041120_to_20041127
    País: N/A
    Duración: N/A días
    Muertos: N/A
    Desplazados: N/A

  Evento: DFO_2625_From_20050211_to_20050226
    País: N/A
    Duración: N/A días
    Muertos: N/A
    Desplazados: N/A

  Evento: DFO_2753_From_20051019_to_20051025
    País: N/A
    Duración: N/A días
    Muertos: N/A
    Desplazados: N/A

  Evento: DFO_2761_From_20050915_to_20051216
    País: N/A
    Duración: N/A días
    Muertos: N/A
    Desplazados: N/A

  Evento: DFO_2823_From_20060322_to_20060602
    País: N/A
    Duración: N/A días
    Muertos: N/A
    Desplazados: N/A

  Evento: DFO_3080_From_20070519_to_20070627
    País: N/A
    Duración: N/A días
    

In [3]:
# --- Análisis refinado: inundación bajo dosel de manglar ---
print("=== ANÁLISIS REFINADO: Inundación bajo dosel ===\n")
print("En manglares, la inundación AUMENTA el backscatter SAR (doble rebote)")
print("SAR diff NEGATIVO = probable inundación bajo dosel\n")

print(f"{'Estación':<20} {'SAR diff (dB)':>15} {'NDVI sept-2020':>15} {'Interpretación':>25}")
print("-" * 77)

# NDVI de septiembre 2020 por estación (datos de Fase 2)
ndvi_sept2020 = {
    'Isla Boquerón': -0.018,
    'Punta Cerro': -0.078,
    'Punta Chino': 0.200,
    'Río Sevilla': 0.050,
    'Caño Palos': 0.400,
    'CP Pajarales': 0.300,
    'Caño Clarín': 0.500,
    'VIPIS': 0.350,
}

for name, coords in stations.items():
    pt = ee.Geometry.Point(coords).buffer(500)
    diff_val = sar_diff.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pt,
        scale=10,
        maxPixels=1e8
    ).get('SAR_diff').getInfo()
    
    ndvi = ndvi_sept2020.get(name, None)
    
    if diff_val and diff_val < -1 and ndvi and ndvi < 0.2:
        interp = "Inundación + degradación"
    elif diff_val and diff_val < -1:
        interp = "Inundación bajo dosel"
    elif ndvi and ndvi < 0:
        interp = "Degradación severa"
    elif ndvi and ndvi < 0.2:
        interp = "Estrés hídrico"
    else:
        interp = "Sin afectación mayor"
    
    diff_str = f"{diff_val:.2f}" if diff_val else "N/A"
    ndvi_str = f"{ndvi:.3f}" if ndvi else "N/A"
    print(f"{name:<20} {diff_str:>15} {ndvi_str:>15} {interp:>25}")

# Mapa de inundación
print("\n=== MAPA DE INUNDACIÓN ===")
import geemap
Map_flood = geemap.Map(center=[10.75, -74.45], zoom=10)
Map_flood.add_basemap('SATELLITE')

# SAR diferencia
vis_sar = {'min': -5, 'max': 5, 'palette': ['blue', 'white', 'red']}
Map_flood.addLayer(sar_diff, vis_sar, 'SAR diferencia (seco - inundación)')

# Inundación agua abierta (diff > 3)
Map_flood.addLayer(flood_mask, {'palette': ['cyan']}, 'Inundación agua abierta', opacity=0.7)

# Inundación bajo dosel (diff < -2)
understory_flood = sar_diff.lt(-2).selfMask().rename('understory')
Map_flood.addLayer(understory_flood, {'palette': ['magenta']}, 'Inundación bajo dosel', opacity=0.7)

# AOI y estaciones
styled_aoi = ee.FeatureCollection([ee.Feature(aoi)]).style(color='FF0000', fillColor='00000000', width=2)
Map_flood.addLayer(styled_aoi, {}, 'AOI')

for name, coords in stations.items():
    pt = ee.Feature(ee.Geometry.Point(coords).buffer(500))
    Map_flood.addLayer(ee.FeatureCollection([pt]).style(color='FFFF00', fillColor='FFFF0080', width=2), {}, name)

# Área inundación bajo dosel
understory_area = understory_flood.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=10,
    maxPixels=1e13
).get('understory').getInfo()

print(f"Área inundación agua abierta: {flood_area/1e6:.1f} km²")
print(f"Área inundación bajo dosel:   {understory_area/1e6:.1f} km²")
print(f"Área total afectada:          {(flood_area + understory_area)/1e6:.1f} km²")

# Guardar datos
import pandas as pd
flood_df = pd.DataFrame({
    'Tipo': ['Agua abierta (SAR diff > 3 dB)', 'Bajo dosel manglar (SAR diff < -2 dB)', 'Total'],
    'Área km²': [round(flood_area/1e6, 1), round(understory_area/1e6, 1), round((flood_area+understory_area)/1e6, 1)]
})
flood_df.to_csv('../outputs/tables/inundacion_sar_sept2020.csv', index=False)
print("\nDatos guardados en outputs/tables/inundacion_sar_sept2020.csv")

Map_flood


=== ANÁLISIS REFINADO: Inundación bajo dosel ===

En manglares, la inundación AUMENTA el backscatter SAR (doble rebote)
SAR diff NEGATIVO = probable inundación bajo dosel

Estación               SAR diff (dB)  NDVI sept-2020            Interpretación
-----------------------------------------------------------------------------
Isla Boquerón                  -0.26          -0.018        Degradación severa
Punta Cerro                    -0.63          -0.078        Degradación severa
Punta Chino                    -0.42           0.200      Sin afectación mayor
Río Sevilla                    -0.76           0.050            Estrés hídrico
Caño Palos                      0.15           0.400      Sin afectación mayor
CP Pajarales                   -1.71           0.300     Inundación bajo dosel
Caño Clarín                    -0.36           0.500      Sin afectación mayor
VIPIS                           0.12           0.350      Sin afectación mayor

=== MAPA DE INUNDACIÓN ===
Área inunda

Map(center=[10.75, -74.45], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [4]:
import ee
import pandas as pd
import numpy as np
import os

ee.Initialize()

aoi_coords = [
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]
aoi = ee.Geometry.Polygon(aoi_coords)
os.makedirs('../outputs/tables', exist_ok=True)

# ===============================================================
# 1. GLOBAL FLOOD DATABASE — Cruce con NDVI por evento
# ===============================================================
print("=" * 70)
print("1. GLOBAL FLOOD DATABASE — Área inundada y cruce con NDVI")
print("=" * 70)

gfd = ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')
gfd_cgsm = gfd.filterBounds(aoi)

# S2 para NDVI
def mask_s2(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2))

# Extraer eventos con área inundada
events_list = gfd_cgsm.toList(50)
n_events = events_list.size().getInfo()

print(f"\nEventos de inundación en CGSM: {n_events}")
print(f"\n{'Evento':<15} {'Inicio':>12} {'Fin':>12} {'Área inund (km²)':>18} {'Duración (días)':>16}")
print("-" * 75)

flood_records = []

for i in range(n_events):
    try:
        img = ee.Image(events_list.get(i))
        idx = img.get('system:index').getInfo()
        
        # Extraer fechas del nombre (DFO_XXXX_From_YYYYMMDD_to_YYYYMMDD)
        parts = idx.split('_')
        start_date = f"{parts[3][:4]}-{parts[3][4:6]}-{parts[3][6:]}"
        end_date = f"{parts[5][:4]}-{parts[5][4:6]}-{parts[5][6:]}"
        
        # Calcular área inundada (banda 'flooded' = 1 donde hubo inundación)
        flood_band = img.select('flooded').clip(aoi)
        flood_area = flood_band.eq(1).multiply(ee.Image.pixelArea()).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=aoi, scale=250, maxPixels=1e13
        ).get('flooded').getInfo()
        flood_km2 = flood_area / 1e6 if flood_area else 0
        
        # Duración
        from datetime import datetime
        d1 = datetime.strptime(start_date, '%Y-%m-%d')
        d2 = datetime.strptime(end_date, '%Y-%m-%d')
        duracion = (d2 - d1).days
        
        print(f"{idx[:15]:<15} {start_date:>12} {end_date:>12} {flood_km2:>18.1f} {duracion:>16}")
        
        flood_records.append({
            'evento': idx,
            'inicio': start_date,
            'fin': end_date,
            'area_inundada_km2': round(flood_km2, 1),
            'duracion_dias': duracion
        })
    except Exception as e:
        print(f"  Error evento {i}: {str(e)[:50]}")

flood_df = pd.DataFrame(flood_records)
flood_df.to_csv('../outputs/tables/gfd_eventos_cgsm.csv', index=False)
print(f"\n{len(flood_records)} eventos guardados en outputs/tables/gfd_eventos_cgsm.csv")

# Resumen
if len(flood_records) > 0:
    print(f"\nResumen:")
    print(f"  Eventos con inundación > 0 km²: {sum(1 for r in flood_records if r['area_inundada_km2'] > 0)}")
    print(f"  Área máxima inundada: {max(r['area_inundada_km2'] for r in flood_records):.1f} km²")
    print(f"  Evento más largo: {max(r['duracion_dias'] for r in flood_records)} días")

# ===============================================================
# 2. JRC GLOBAL SURFACE WATER — Transiciones y estacionalidad
# ===============================================================
print("\n" + "=" * 70)
print("2. JRC GLOBAL SURFACE WATER — Transiciones y dinámica hídrica")
print("=" * 70)

jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').clip(aoi)

# Banda 'transition' — cambios entre períodos
transition = jrc.select('transition')
occurrence = jrc.select('occurrence')
seasonality = jrc.select('seasonality')

# Clases de transición JRC
# 1=Permanent, 2=New permanent, 3=Lost permanent, 4=Seasonal,
# 5=New seasonal, 6=Lost seasonal, 7=Seasonal to permanent,
# 8=Permanent to seasonal, 9=Ephemeral permanent, 10=Ephemeral seasonal

trans_classes = {
    1: 'Permanente',
    2: 'Nuevo permanente', 
    3: 'Permanente perdido',
    4: 'Estacional',
    5: 'Nuevo estacional',
    6: 'Estacional perdido',
    7: 'Estacional → permanente',
    8: 'Permanente → estacional',
    9: 'Efímero permanente',
    10: 'Efímero estacional'
}

print(f"\n{'Clase de transición':<30} {'Área (km²)':>12}")
print("-" * 44)

trans_records = []
for code, name in trans_classes.items():
    area = transition.eq(code).multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
    ).get('transition').getInfo()
    area_km2 = area / 1e6 if area else 0
    if area_km2 > 0:
        print(f"{name:<30} {area_km2:>12.1f}")
        trans_records.append({'clase': name, 'codigo': code, 'area_km2': round(area_km2, 1)})

trans_df = pd.DataFrame(trans_records)
trans_df.to_csv('../outputs/tables/jrc_transiciones_cgsm.csv', index=False)

# Estacionalidad — cuántos meses al año hay agua
print(f"\n{'Estacionalidad':<30} {'Área (km²)':>12}")
print("-" * 44)
for meses in [1, 3, 6, 9, 12]:
    area = seasonality.gte(meses).multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
    ).get('seasonality').getInfo()
    area_km2 = area / 1e6 if area else 0
    print(f"Agua ≥ {meses} meses/año{'':<15} {area_km2:>12.1f}")

# Agua permanente vs estacional
perm = occurrence.gt(75).multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).get('occurrence').getInfo()
seas = occurrence.gt(25).And(occurrence.lte(75)).multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).get('occurrence').getInfo()

print(f"\nAgua permanente (>75%): {perm/1e6:.1f} km²")
print(f"Agua estacional (25-75%): {seas/1e6:.1f} km²")

# ===============================================================
# 3. Cruce: transiciones JRC en zonas de manglar
# ===============================================================
print("\n" + "=" * 70)
print("3. CRUCE: Transiciones JRC en zonas de manglar")
print("=" * 70)

# Manglar según nuestra clasificación
def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

s2_idx = s2.map(add_indices)
comp = s2_idx.filterDate('2020-01-01', '2020-12-31').median().clip(aoi)
ndvi = comp.select('NDVI')
cmri = comp.select('CMRI')
manglar_zone = cmri.gt(0.3).And(ndvi.gt(0.4))

# Transiciones dentro de zonas de manglar
print(f"\n{'Transición en zona manglar':<35} {'Área (km²)':>12}")
print("-" * 49)

for code, name in trans_classes.items():
    area = transition.eq(code).And(manglar_zone).multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
    ).get('transition').getInfo()
    area_km2 = area / 1e6 if area else 0
    if area_km2 > 0.1:
        print(f"{name:<35} {area_km2:>12.1f}")

# ===============================================================
# 4. Cruce: Eventos GFD en estaciones de monitoreo
# ===============================================================
print("\n" + "=" * 70)
print("4. CRUCE: Eventos históricos de inundación por estación")
print("=" * 70)

stations = {
    'Isla Boquerón': [-74.298, 10.962],
    'Punta Cerro': [-74.283, 10.973],
    'Punta Chino': [-74.305, 10.912],
    'Río Sevilla': [-74.325, 10.880],
    'Caño Palos': [-74.471, 10.758],
    'CP Pajarales': [-74.75, 10.85],
    'Caño Clarín': [-74.55, 10.55],
    'VIPIS': [-74.65, 11.02],
}

print(f"\n{'Estación':<20} {'Eventos con inundación':>25} {'Evento más severo':>20}")
print("-" * 67)

for name, coords in stations.items():
    pt = ee.Geometry.Point(coords).buffer(500)
    n_inundado = 0
    max_flood = 0
    max_event = ''
    
    for i in range(min(n_events, 16)):
        try:
            img = ee.Image(events_list.get(i))
            idx = img.get('system:index').getInfo()
            flood_val = img.select('flooded').reduceRegion(
                reducer=ee.Reducer.max(), geometry=pt, scale=250, maxPixels=1e8
            ).get('flooded').getInfo()
            
            if flood_val and flood_val > 0:
                n_inundado += 1
                if flood_val > max_flood:
                    max_flood = flood_val
                    max_event = idx[:20]
        except:
            pass
    
    print(f"{name:<20} {n_inundado:>25} {max_event:>20}")

print("\n=== ANÁLISIS NASA COMPLETO ===")
print("Archivos guardados:")
print("  outputs/tables/gfd_eventos_cgsm.csv")
print("  outputs/tables/jrc_transiciones_cgsm.csv")

1. GLOBAL FLOOD DATABASE — Área inundada y cruce con NDVI

Eventos de inundación en CGSM: 16

Evento                Inicio          Fin   Área inund (km²)  Duración (días)
---------------------------------------------------------------------------
DFO_1818_From_2   2001-10-26   2001-11-15                0.0               20
DFO_1996_From_2   2002-07-20   2002-07-31              959.5               11
DFO_2588_From_2   2004-11-20   2004-11-27             1027.2                7
DFO_2625_From_2   2005-02-11   2005-02-26             1149.9               15
DFO_2753_From_2   2005-10-19   2005-10-25                0.0                6
DFO_2761_From_2   2005-09-15   2005-12-16             1063.7               92
DFO_2823_From_2   2006-03-22   2006-06-02               68.4               72
DFO_3080_From_2   2007-05-19   2007-06-27               57.5               39
DFO_3212_From_2   2007-10-01   2007-12-10             1025.7               70
DFO_3311_From_2   2008-05-27   2008-05-28         

In [5]:
import pandas as pd
import os

os.makedirs('../outputs/tables', exist_ok=True)

# Validación final
final = pd.DataFrame({
    'Métrica': [
        'Precisión global (OA)', "Precisión (Producer's)", "Recall (User's)", 'F1-score',
        'Área GMW 2020 (km²)', 'Área clasificación (km²)', 'Ratio área',
        'TP', 'TN', 'FP', 'FN',
        'Umbral CMRI', 'Umbral NDVI',
        'Restricción elevación', 'Restricción distancia agua'
    ],
    'Valor': [
        0.899, 0.428, 0.457, 0.442,
        376.5, 556.0, round(556.0/376.5, 2),
        55, 1289, 84, 72,
        '> 0.3', '> 0.70',
        '< 10 m (SRTM)', '< 3 km (JRC)'
    ]
})
final.to_csv('../outputs/tables/validacion_gmw_final.csv', index=False)

# Eventos por estación
estaciones_flood = pd.DataFrame({
    'Estación': ['Isla Boquerón', 'Punta Cerro', 'Punta Chino', 'Río Sevilla',
                 'Caño Palos', 'CP Pajarales', 'Caño Clarín', 'VIPIS'],
    'Eventos inundación (2001-2017)': [10, 10, 9, 10, 2, 9, 1, 9],
    'Evento más severo': ['DFO_1996 (jul 2002)', 'DFO_1996 (jul 2002)',
                          'DFO_1996 (jul 2002)', 'DFO_1996 (jul 2002)',
                          'DFO_2588 (nov 2004)', 'DFO_2625 (feb 2005)',
                          'DFO_4495 (ago 2017)', 'DFO_1996 (jul 2002)']
})
estaciones_flood.to_csv('../outputs/tables/eventos_inundacion_estaciones.csv', index=False)

print("Archivos guardados:")
print("  validacion_gmw_final.csv")
print("  eventos_inundacion_estaciones.csv")
print(final.to_string(index=False))

Archivos guardados:
  validacion_gmw_final.csv
  eventos_inundacion_estaciones.csv
                   Métrica         Valor
     Precisión global (OA)         0.899
    Precisión (Producer's)         0.428
           Recall (User's)         0.457
                  F1-score         0.442
       Área GMW 2020 (km²)         376.5
  Área clasificación (km²)         556.0
                Ratio área          1.48
                        TP            55
                        TN          1289
                        FP            84
                        FN            72
               Umbral CMRI         > 0.3
               Umbral NDVI        > 0.70
     Restricción elevación < 10 m (SRTM)
Restricción distancia agua  < 3 km (JRC)
